# Legal Outcome Prediction Pipeline
**Runtime → Change runtime type → T4 GPU** before running.

Steps:
1. Upload the `NLPPW` project folder to your Google Drive (e.g. `My Drive/NLPPW`)
2. Add your HuggingFace token as a Colab secret named `HF_TOKEN` (🔑 icon in the left sidebar)
3. Run all cells in order

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT_PATH = '/content/drive/MyDrive/NLPPW'  # adjust if you placed it elsewhere
sys.path.insert(0, PROJECT_PATH)

import os
os.chdir(PROJECT_PATH)
print('Working directory:', os.getcwd())

In [ ]:
%%capture
!pip install \
    'transformers>=4.40.0' \
    'sentence-transformers>=2.7.0' \
    'datasets>=2.19.0' \
    'huggingface-hub' \
    'scikit-learn>=1.4.0' \
    'interpret>=0.6.0' \
    'numpy>=1.26.0' \
    'pandas>=2.2.0' \
    'matplotlib>=3.8.0' \
    'seaborn>=0.13.0' \
    'tqdm>=4.66.0' \
    'joblib>=1.4.0'
print('Dependencies installed.')

In [ ]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('HF_TOKEN set.')

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(torch.cuda.get_device_name(0))

In [ ]:
import config

# ── Sample limits: reduce for a quick demo, set to None for the full dataset ──
config.MAX_TRAIN_SAMPLES = 500
config.MAX_VAL_SAMPLES   = 100
config.MAX_TEST_SAMPLES  = 100

print(f'Train: {config.MAX_TRAIN_SAMPLES}, Val: {config.MAX_VAL_SAMPLES}, Test: {config.MAX_TEST_SAMPLES}')
print(f'Stage 1 model : {config.LEGALBERT_MODEL}')
print(f'Stage 2 model : {config.SENTENCE_TRANSFORMER_MODEL}')
print(f'Classifier    : {config.CLASSIFIER_TYPE}')
print(f'Fact negatives: {config.FACT_NEGATIVES}  (sim threshold: {config.FACT_SIM_THRESHOLD})')
print(f'Dynamic top-k : {config.DYNAMIC_TOPK}  (alpha: {config.DYNAMIC_TOPK_ALPHA}, floor: {config.PREMISE_FLOOR})')

In [ ]:
import zipfile
from pathlib import Path
import config

# Extract dataset to local Colab storage — much faster than reading
# 12 500+ small JSON files directly from Drive.
ZIP_PATH       = Path(PROJECT_PATH) / "echr-args-dataset.zip"
EXTRACT_TO     = Path("/content/echr-args-dataset")

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Zip file not found at {ZIP_PATH}\n"
        "Upload echr-args-dataset.zip to your NLPPW folder on Drive."
    )

if not EXTRACT_TO.exists():
    print(f"Extracting {ZIP_PATH.name} → {EXTRACT_TO} ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_TO)
    print("Done.")
else:
    print(f"Dataset already extracted at {EXTRACT_TO} — skipping.")

# Point config to the extracted folder
config.NEW_DATASET_DIR = EXTRACT_TO
n_files = len(list(EXTRACT_TO.glob("*.json")))
print(f"Dataset ready: {n_files} JSON files in {EXTRACT_TO}")

## Stage 1 — Fine-tune LegalBERT (run once)
Fine-tunes LegalBERT on the 12 500+ case ECtHR dataset extracted above.
Splitting is done at the **case_id level** so no case leaks across train/val/test.

**Similarity-filtered fact negatives** (`config.FACT_NEGATIVES`): before fine-tuning, fact sentences
that are dissimilar to any party argument (cosine sim < `FACT_SIM_THRESHOLD`) are added as NON_PREMISE.
Run the inspection cell below to review what gets kept vs filtered.

The checkpoint is saved to `outputs/stage1_legalbert/checkpoint-best` on your Drive.
**Skip fine-tuning on subsequent runs** — `config.py` auto-detects the saved checkpoint.

In [ ]:
from stage1_argument_mining.fact_filter import inspect_fact_negatives

inspect_fact_negatives(n_cases=5, n_sentences=5)

In [ ]:
import os
from pathlib import Path
import config

checkpoint = config.OUTPUT_DIR / "stage1_legalbert" / "checkpoint-best"

if checkpoint.exists():
    print(f"Fine-tuned checkpoint already found at:\n  {checkpoint}")
    print("Skipping fine-tuning — delete that folder to retrain.")
else:
    print("No checkpoint found. Starting fine-tuning on ECHR Argumentation Corpus...")
    from stage1_argument_mining.finetune_legalbert import finetune
    finetune()
    # Reload config so LEGALBERT_MODEL picks up the new checkpoint
    import importlib
    importlib.reload(config)
    print(f"\nLEGALBERT_MODEL is now: {config.LEGALBERT_MODEL}")

In [ ]:
# Evaluate the Stage 1 checkpoint on test set
import numpy as np
from sklearn.metrics import classification_report, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from stage1_argument_mining.finetune_legalbert import prepare_data, ID2LABEL, LABEL2ID

checkpoint = config.OUTPUT_DIR / "stage1_legalbert" / "checkpoint-best"

if checkpoint.exists():
    print(f"Evaluating checkpoint: {checkpoint}")
    
    # Load test data
    _, _, test_rows = prepare_data()
    
    # Tokenize
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    test_ds = Dataset.from_dict({
        "text": [r["text"] for r in test_rows],
        "label": [r["label"] for r in test_rows],
    })
    
    def tokenize_batch(batch):
        return tokenizer(batch["text"], truncation=True, max_length=config.S1_MAX_SEQ_LEN, padding="max_length")
    
    test_ds = test_ds.map(tokenize_batch, batched=True, desc="Tokenizing test set")
    
    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID)
    
    # Evaluate
    trainer = Trainer(
        model=model,
        args=TrainingArguments(output_dir=str(config.OUTPUT_DIR / "temp"), per_device_eval_batch_size=config.S1_BATCH_SIZE, seed=config.RANDOM_SEED)
    )
    
    out = trainer.predict(test_ds)
    preds = np.argmax(out.predictions, axis=-1)
    
    print("\n" + "="*60)
    print("Stage 1 LegalBERT - Test Set Results")
    print("="*60)
    print(classification_report(out.label_ids, preds, target_names=["NON_PREMISE", "PREMISE"]))
    
    f1_macro = f1_score(out.label_ids, preds, average="macro", zero_division=0)
    f1_binary = f1_score(out.label_ids, preds, average="binary", zero_division=0)
    print(f"\nMacro F1:  {f1_macro:.4f}")
    print(f"Binary F1: {f1_binary:.4f}")
    print("="*60)
else:
    print("No checkpoint found. Run fine-tuning first.")

## Stage 1 — Argument Mining
Loads LegalBERT and classifies each sentence as a premise or not.

In [ ]:
from pathlib import Path
from data.data_loader import get_dataset
from stage1_argument_mining.sequence_filter import run_stage1, load_stage1, print_stage1_stats

dataset = get_dataset()

if Path(config.STAGE1_CACHE).exists():
    print(f"Cached premises found at {config.STAGE1_CACHE} — skipping extraction.")
    print("Delete this file to re-run Stage 1 extraction.")
    stage1_output = load_stage1()
else:
    from stage1_argument_mining.argument_extractor import LegalBERTArgumentExtractor
    extractor = LegalBERTArgumentExtractor()
    stage1_output = run_stage1(dataset, extractor)

print_stage1_stats(stage1_output)

In [ ]:
from stage1_argument_mining.argument_extractor import _split_sentences, LegalBERTArgumentExtractor

print("Stage 1 model:", config.LEGALBERT_MODEL)
print(f"Dynamic top-k: {config.DYNAMIC_TOPK}  (alpha={config.DYNAMIC_TOPK_ALPHA}, floor={config.PREMISE_FLOOR})")
print()

# Build extractor only for diagnostics if not already loaded
if 'extractor' not in dir() or extractor is None:
    extractor = LegalBERTArgumentExtractor()

sample_paragraphs = dataset["test"][0]["paragraphs"]
print(f"Probing first test case ({len(sample_paragraphs)} paragraphs), raw scores:\n")
for para in sample_paragraphs[:3]:
    for sent in _split_sentences(para):
        is_p, conf = extractor.predict_sentence(sent)
        label = "PREMISE" if is_p else "---"
        print(f"  {conf:.3f}  {label:<10}  {sent[:90]}")

premises = extractor.extract_premises(sample_paragraphs)
print(f"\nDynamic top-k selected {len(premises)} premises:")
for p in premises:
    print(f"  {p['confidence']:.3f}  [para {p['paragraph_id']}]  {p['sentence'][:120]}")

## Stage 2 — Outcome Prediction
Embeds extracted premises with Sentence-BERT and trains the classifier.

In [ ]:
from pathlib import Path
from stage2_outcome_prediction.embedder import PremiseEmbedder
from stage2_outcome_prediction.classifier import (
    train_classifier, save_classifier, load_classifier, quick_eval)

embedder = PremiseEmbedder()

X_train, y_train = embedder.prepare_split(stage1_output['train'])
X_val,   y_val   = embedder.prepare_split(stage1_output['val'])
X_test,  y_test  = embedder.prepare_split(stage1_output['test'])

if Path(config.MODEL_CACHE).exists():
    print(f"Cached classifier found at {config.MODEL_CACHE} — skipping training.")
    print("Delete this file to retrain.")
    clf = load_classifier()
else:
    clf = train_classifier(X_train, y_train)
    save_classifier(clf)

quick_eval(clf, X_val, y_val)

## 🧪 EXPERIMENTAL: Hybrid Embedding Approach

The cells below demonstrate the **hybrid embedding experiment**: instead of embedding only extracted premise sentences, we embed **all paragraphs** with premise-awareness features.

**Key differences:**
- **Baseline (above)**: Embeds only premise sentences → 2309 features
- **Hybrid (below)**: Embeds full paragraphs with premise flags → 2313 features

**Hypothesis**: Non-premise text (procedural history, factual background) may provide useful context that improves predictions.

Run both approaches and compare F1 scores!

In [ ]:
# Enable hybrid mode
import config
config.USE_HYBRID_EMBEDDER = True
print(f"Hybrid mode: {config.USE_HYBRID_EMBEDDER}")
print(f"Premise weight boost: {config.HYBRID_PREMISE_WEIGHT_BOOST}")

In [ ]:
from pathlib import Path
from stage2_outcome_prediction.embedder_hybrid import HybridPremiseEmbedder
from stage2_outcome_prediction.classifier import (
    train_classifier, save_classifier, load_classifier, quick_eval)

embedder_hybrid = HybridPremiseEmbedder()
print("Using HYBRID embedder (full paragraphs + premise features)")

X_train_hybrid, y_train_hybrid = embedder_hybrid.prepare_split(stage1_output['train'])
X_val_hybrid,   y_val_hybrid   = embedder_hybrid.prepare_split(stage1_output['val'])
X_test_hybrid,  y_test_hybrid  = embedder_hybrid.prepare_split(stage1_output['test'])

print(f"\nFeature dimensions:")
print(f"  Premise-only: {X_train.shape[1]} features")
print(f"  Hybrid:       {X_train_hybrid.shape[1]} features")

# Save hybrid classifier with different cache name
HYBRID_MODEL_CACHE = config.OUTPUT_DIR / "stage2_classifier_hybrid.joblib"

if Path(HYBRID_MODEL_CACHE).exists():
    print(f'Cached hybrid classifier found at {HYBRID_MODEL_CACHE} — skipping training.')
    print('Delete this file to retrain.')
    clf_hybrid = load_classifier(HYBRID_MODEL_CACHE)
else:
    clf_hybrid = train_classifier(X_train_hybrid, y_train_hybrid)
    save_classifier(clf_hybrid, HYBRID_MODEL_CACHE)

quick_eval(clf_hybrid, X_val_hybrid, y_val_hybrid)

## Evaluation

In [ ]:
from stage2_outcome_prediction.classifier import predict, predict_with_thresholds, tune_thresholds
from evaluation.metrics import (compare_classifiers, per_article_f1,
                                 print_per_article_f1, train_baseline_classifier)
from evaluation.qualitative_review import run_qualitative_review
from data.data_loader import ARTICLE_NAMES

print("="*70)
print("THREE-WAY EVALUATION: Baseline vs Premise vs Hybrid")
print("="*70)

# 1. Baseline (raw text, no premise extraction)
print("\n[1/3] Training baseline (raw text)...")
y_pred_baseline, clf_baseline = train_baseline_classifier(
    stage1_output['train'], stage1_output['test'], embedder)

# 2. Premise (current approach)
print("\n[2/3] Evaluating premise classifier...")
thresholds_premise = tune_thresholds(clf, X_val, y_val)
print("  Tuned thresholds:")
for name, t in zip(ARTICLE_NAMES, thresholds_premise):
    print(f"    {name:<12} threshold={t:+.4f}")
y_pred_premise = predict_with_thresholds(clf, X_test, thresholds_premise)

# 3. Hybrid (full-text with premise features)
print("\n[3/3] Evaluating hybrid classifier...")
thresholds_hybrid = tune_thresholds(clf_hybrid, X_val_hybrid, y_val_hybrid)
y_pred_hybrid = predict_with_thresholds(clf_hybrid, X_test_hybrid, thresholds_hybrid)

# Overall metrics comparison
print("\n" + "="*70)
print("OVERALL METRICS")
print("="*70)
print(f"{'Metric':<22} {'Baseline':>15} {'Premise':>15} {'Hybrid':>15}")
print("="*70)

from evaluation.metrics import compute_metrics
metrics_baseline = compute_metrics(y_test, y_pred_baseline)
metrics_premise = compute_metrics(y_test, y_pred_premise)
metrics_hybrid = compute_metrics(y_test_hybrid, y_pred_hybrid)

for metric in ['macro_f1', 'micro_f1', 'macro_precision', 'macro_recall', 'micro_precision', 'micro_recall']:
    b = metrics_baseline[metric]
    p = metrics_premise[metric]
    h = metrics_hybrid[metric]
    print(f"{metric:<22} {b:>15.4f} {p:>15.4f} {h:>15.4f}")
print("="*70)

# Per-article comparison
pa_f1_baseline = per_article_f1(y_test, y_pred_baseline)
pa_f1_premise = per_article_f1(y_test, y_pred_premise)
pa_f1_hybrid = per_article_f1(y_test_hybrid, y_pred_hybrid)

baseline_dict = {r['article']: r['f1'] for r in pa_f1_baseline}
premise_dict = {r['article']: r['f1'] for r in pa_f1_premise}
hybrid_dict = {r['article']: r['f1'] for r in pa_f1_hybrid}

print("\n" + "="*80)
print("PER-ARTICLE F1 SCORES")
print("="*80)
print(f"{'Article':<16} {'Baseline':>15} {'Premise':>15} {'Hybrid':>15} {'Support':>10}")
print("="*80)

for r in pa_f1_baseline:
    article = r['article']
    b = baseline_dict.get(article, 0.0)
    p = premise_dict.get(article, 0.0)
    h = hybrid_dict.get(article, 0.0)
    support = r['support']
    print(f"{article:<16} {b:>15.4f} {p:>15.4f} {h:>15.4f} {support:>10}")
print("="*80)

# Determine best approach
best = max([('Baseline', metrics_baseline['macro_f1']),
            ('Premise', metrics_premise['macro_f1']),
            ('Hybrid', metrics_hybrid['macro_f1'])], key=lambda x: x[1])
print(f"\nBest approach: {best[0]} (macro F1: {best[1]:.4f})")

# Qualitative review with premise classifier
print("\n" + "="*70)
print("QUALITATIVE REVIEW (Premise Classifier)")
print("="*70)
run_qualitative_review(stage1_output['test'], X_test, clf, embedder)

In [ ]:
# Premise Count Analysis - Classical ML
from evaluation.premise_count_analysis import (
    group_cases_by_premise_count, plot_premise_count_analysis, print_premise_count_table)

print("\n" + "="*70)
print("PREMISE COUNT ANALYSIS - Classical ML (SVM)")
print("="*70)

predictions_classical = {
    'Baseline': y_pred_baseline,
    'Premise': y_pred_premise,
    'Hybrid': y_pred_hybrid
}

results_classical = group_cases_by_premise_count(stage1_output['test'], predictions_classical)

print("\nPerformance by number of extracted premises:")
print_premise_count_table(results_classical)

plot_premise_count_analysis(
    results_classical,
    title="Classical ML: Performance vs Premise Count",
    output_path=config.OUTPUT_DIR / "premise_count_analysis_classical.png",
    colors={'Baseline': '#1f77b4', 'Premise': '#ff7f0e', 'Hybrid': '#2ca02c'}
)

## LegalBERT Classifier Comparison
Validates that Stage 1 premise extraction adds value: fine-tunes LegalBERT as a multi-label classifier on **extracted premises only** vs **full text**, then compares both against the explainable pipeline.

This is an additional experiment — our main pipeline remains the explainable SVM/DT/EBM approach.

In [ ]:
from stage2_outcome_prediction.bert_classifier import train_bert_classifier, predict_bert
from evaluation.metrics import compute_metrics, per_article_f1, EVAL_ARTICLE_NAMES
import numpy as np

bert_results = {}
y_test_bert = np.array([c['labels_binary'] for c in stage1_output['test']])

print("="*70)
print("LEGALBERT COMPARISON: Full-text vs Premises vs Hybrid")
print("="*70)

# --- Full-text LegalBERT ---
print('\n[1/3] Training LegalBERT on full text...')
model_full, tok_full = train_bert_classifier(
    stage1_output['train'], stage1_output['val'],
    use_premises=False, use_hybrid=False, epochs=3, batch_size=16)
y_pred_full = predict_bert(model_full, tok_full, stage1_output['test'], 
                           use_premises=False, use_hybrid=False)
bert_results['fulltext_bert'] = compute_metrics(y_test_bert, y_pred_full)
bert_results['fulltext_bert']['per_article'] = per_article_f1(y_test_bert, y_pred_full)
print(f'  Full-text LegalBERT  macro_f1={bert_results["fulltext_bert"]["macro_f1"]:.4f}  '
      f'micro_f1={bert_results["fulltext_bert"]["micro_f1"]:.4f}')

# --- Premises-only LegalBERT ---
print('\n[2/3] Training LegalBERT on extracted premises...')
model_prem, tok_prem = train_bert_classifier(
    stage1_output['train'], stage1_output['val'],
    use_premises=True, use_hybrid=False, epochs=3, batch_size=16)
y_pred_prem = predict_bert(model_prem, tok_prem, stage1_output['test'], 
                          use_premises=True, use_hybrid=False)
bert_results['premises_bert'] = compute_metrics(y_test_bert, y_pred_prem)
bert_results['premises_bert']['per_article'] = per_article_f1(y_test_bert, y_pred_prem)
print(f'  Premises LegalBERT   macro_f1={bert_results["premises_bert"]["macro_f1"]:.4f}  '
      f'micro_f1={bert_results["premises_bert"]["micro_f1"]:.4f}')

# --- Hybrid LegalBERT ---
print('\n[3/3] Training LegalBERT on HYBRID (full text with [PREMISE] markers)...')
model_hybrid, tok_hybrid = train_bert_classifier(
    stage1_output['train'], stage1_output['val'],
    use_premises=False, use_hybrid=True, epochs=3, batch_size=16)
y_pred_hybrid = predict_bert(model_hybrid, tok_hybrid, stage1_output['test'], 
                            use_premises=False, use_hybrid=True)
bert_results['hybrid_bert'] = compute_metrics(y_test_bert, y_pred_hybrid)
bert_results['hybrid_bert']['per_article'] = per_article_f1(y_test_bert, y_pred_hybrid)
print(f'  Hybrid LegalBERT     macro_f1={bert_results["hybrid_bert"]["macro_f1"]:.4f}  '
      f'micro_f1={bert_results["hybrid_bert"]["micro_f1"]:.4f}')

# --- Three-way comparison table ---
print(f'\n{"="*70}')
print("OVERALL METRICS")
print(f'{"="*70}')
print(f'{"Metric":<22} {"Full-text":>15} {"Premises":>15} {"Hybrid":>15}')
print(f'{"="*70}')
for m in ['macro_f1', 'micro_f1', 'macro_precision', 'macro_recall', 'hamming_loss']:
    f = bert_results['fulltext_bert'][m]
    p = bert_results['premises_bert'][m]
    h = bert_results['hybrid_bert'][m]
    print(f'{m:<22} {f:>15.4f} {p:>15.4f} {h:>15.4f}')
print(f'{"="*70}')

print(f'\n{"="*70}')
print("PER-ARTICLE F1 SCORES")
print(f'{"="*70}')
print(f'{"Article":<16} {"Full-text":>15} {"Premises":>15} {"Hybrid":>15}')
print(f'{"="*70}')
for name in EVAL_ARTICLE_NAMES:
    pa_f = {r['article']: r['f1'] for r in bert_results['fulltext_bert'].get('per_article', [])}
    pa_p = {r['article']: r['f1'] for r in bert_results['premises_bert'].get('per_article', [])}
    pa_h = {r['article']: r['f1'] for r in bert_results['hybrid_bert'].get('per_article', [])}
    print(f'{name:<16} {pa_f.get(name, 0.0):>15.4f} {pa_p.get(name, 0.0):>15.4f} {pa_h.get(name, 0.0):>15.4f}')
print(f'{"="*70}')

# Determine best approach
best = max([('Full-text', bert_results['fulltext_bert']['macro_f1']),
            ('Premises', bert_results['premises_bert']['macro_f1']),
            ('Hybrid', bert_results['hybrid_bert']['macro_f1'])], key=lambda x: x[1])
print(f'\nBest LegalBERT approach: {best[0]} (macro F1: {best[1]:.4f})')

In [ ]:
# Premise Count Analysis - LegalBERT
from evaluation.premise_count_analysis import (
    group_cases_by_premise_count, plot_premise_count_analysis, print_premise_count_table)

print("\n" + "="*70)
print("PREMISE COUNT ANALYSIS - LegalBERT (Deep Learning)")
print("="*70)

predictions_bert = {
    'Full-text': y_pred_full,
    'Premises': y_pred_prem,
    'Hybrid': y_pred_hybrid
}

results_bert = group_cases_by_premise_count(stage1_output['test'], predictions_bert)

print("\nPerformance by number of extracted premises:")
print_premise_count_table(results_bert)

plot_premise_count_analysis(
    results_bert,
    title="LegalBERT: Performance vs Premise Count",
    output_path=config.OUTPUT_DIR / "premise_count_analysis_bert.png",
    colors={'Full-text': '#1f77b4', 'Premises': '#ff7f0e', 'Hybrid': '#2ca02c'}
)

### 🧪 BERT + Hybrid Approach

Test whether premise-awareness helps even with a deep learning model. This trains LegalBERT on full text with **[PREMISE] markers** around paragraphs containing extracted premises.

In [ ]:
from data.data_loader import ARTICLE_NAMES
import numpy as np

def predict_case(paragraphs, extractor, embedder, clf, labels_binary=None):
    """Run the full pipeline on a raw list of paragraph strings and print predicted articles."""
    premises = extractor.extract_premises(paragraphs)
    used_fallback = len(premises) == 0

    if used_fallback:
        premise_text = " ".join(paragraphs)
        premises_for_embed = []
    else:
        premise_text = " ".join(p["sentence"] for p in premises)
        premises_for_embed = premises

    case = {
        "paragraphs":    paragraphs,
        "premises":      premises_for_embed,
        "premise_text":  premise_text,
        "used_fallback": used_fallback,
        "labels_binary": labels_binary if labels_binary is not None else [0] * len(ARTICLE_NAMES),
    }

    X, _ = embedder.prepare_split([case])
    y_pred = clf.predict(X)[0]

    predicted = [ARTICLE_NAMES[i] for i, v in enumerate(y_pred) if v == 1]

    print(f"Premises extracted : {len(premises)}  (fallback={'yes' if used_fallback else 'no'})")
    print(f"Predicted articles : {predicted or ['No violation']}")

    if labels_binary is not None:
        true = [ARTICLE_NAMES[i] for i, v in enumerate(labels_binary) if v == 1]
        print(f"True articles      : {true or ['No violation']}")

    if premises:
        print("\nTop premises used:")
        for p in premises[:5]:
            print(f"  ({p['confidence']:.3f}) {p['sentence'][:120]}")

    return predicted


# ── paste your paragraph list here ──────────────────────────────────────────
paragraphs = [
"5. The applicant was born in 1940 and lives in Odesa.",
"6. At the time of the events the applicant was the director general of a joint venture V. ("company V."), which had its office in the premises belonging to a joint stock company Y. ("company Y.").",
"7. In March 2001 the owner of company Y. changed. The new management questioned the legality of the use of its premises by company V. More specifically, they challenged the lease contract of 12 January 1999 in respect of those premises, which had been signed by the applicant, on the one side, and N., the chairman of the board of directors of company Y. at the time, on the other side. Under that contract, company V. could use the office space in question from 12 January 1999 to 12 January 2020 without any payment, but in exchange for certain services for company Y.",
"8. Starting from April 2001, company Y. no longer allowed access to its premises to company V. As a result, the applicant transformed his flat in a temporary office of company V.",
"9. In June 2001 company V. brought commercial proceedings against company Y. seeking compliance with the lease contract. Company Y., in turn, lodged a counter-claim seeking invalidation of that contract. By a final decision of the Supreme Court of 25 September 2003, the national courts rejected the claim of company V. and discontinued the proceedings as regards company Y.'s counter-claim. It was concluded that "there [was] no subject matter of the dispute", given that the impugned contract failed to stipulate basic terms inherent in a lease contract and could not therefore be regarded as a lease contract.",
"10. On 3 October 2001 a criminal case was opened in respect of suspected forgery of the lease contract of 12 January 1999, without being targeted against any particular persons.",
"11. On 8 October 2001 the Odesa Prymorskyy District Prosecutor's Office ("the Prymorskyy Prosecutor's Office") issued a warrant for seizure of fifteen documents relevant for the investigation, such as the original of the lease contract itself, related correspondence and several statements of acceptance of the services indicated in the contract (see paragraph 7 above). The seizure was to be carried out in company V.'s office.",
"12. On 11 October 2001 the seizure took place in the applicant's flat, in the presence of his wife. It appears that the applicant was not present. Eleven of the fifteen documents listed in the warrant were seized. The seizure report did not contain any information as to whether it had been handed to any person occupying the premises. The applicant did not specify in the domestic proceedings, or in the present proceedings, how the seizure of the documents had taken place.",
"13. On 15 October 2001 the seizure warrant of 8 October 2001 was served on the applicant.",
"14. On 22 October 2001 the investigator decided that a forensic expert examination of the signatures on the contract of 12 January 1999 was required in order to establish their real date.",
"15. On 7 November 2001 the Odesa Prymorskyy District Court ("the Prymorskyy Court") ordered a search of the applicant's flat, which was also company V.'s office, with a view to collecting samples of his handwriting and signatures. As stated in the court's ruling, "notebooks, correspondence and other personal records with [the applicant's] handwriting" were required for the above-mentioned expert evaluation. That decision was not amenable to appeal.",
"16. On the following day the search took place in the applicant's flat in his presence and resulted in a seizure of eleven documents. The applicant did not provide any description, be it in the domestic proceedings or in the present proceedings, as to how the search had been carried out.",
"17. On 24 December 2001 the investigator ordered a seizure of company V.'s constituent documents from the company's office. It appears that the seizure was carried out on the same day in the applicant's flat.",
"18. On 9 January 2002 the above seizure warrant was served on the applicant.",
"19. On 6 May 2002 company V. founders' meeting decided to suspend the applicant from the exercise of his duties as its director general pending the ongoing criminal proceedings.",
"20. On 20 August 2002 the prosecutor discontinued the proceedings for the absence of unequivocal evidence of a criminal offence. Although a forensic expert examination had established that the signatures on the impugned contract had been antedated (namely, it was established that they had been made no earlier than in February 2001), the official approval of the technical methods used by the expert was previewed only for the autumn of 2002.",
"21. On 1 September 2002 the applicant resumed his duties in company V.",
"22. On 28 August 2002 the applicant brought proceedings against the Prymorskyy Prosecutor's Office claiming compensation in respect of non-pecuniary damage allegedly caused by its unlawful actions. The applicant based his lawsuit on the fact that the criminal proceedings had been terminated, without raising any specific complaints about the search and seizures. He contended that the institution of the criminal proceedings had been arbitrary, which had led, inter alia, to the unlawful search of his flat and the seizure of documents.",
"23. On 9 December 2002 the Prymorskyy Court rejected the applicant's claim as unfounded. The case file does not contain a copy of that decision. It appears that the court's conclusion was that the applicant had not suffered any non-pecuniary damage.",
"24. The applicant appealed. He argued, in particular, that the impugned measures had been devoid of any legitimate purpose given the impossibility at the time to carry out the forensic handwriting examination ordered by the investigator. He further submitted that, in ordering the seizure of documents, no differentiation had been made between the company's premises and his home. The applicant maintained that the first-instance court had left those matters without consideration.",
"25. On 11 September 2003 the Odesa Regional Court of Appeal rejected the applicant's appeal. It held, in particular, that the company's office had de facto been located in the applicant's flat. As regards his complaint about the court's failure to assess all the circumstances of the case, the appellate court dismissed it as ungrounded.",
"26. On 3 February 2006 the Supreme Court upheld the lower courts' decisions."
]

predict_case(paragraphs, extractor, embedder, clf, labels_binary=None)